# Tech Challenge - Fase 3
## Engenharia de Dados e Analytics - State of Data Brasil 2023 a 2025

Notebook consolidado para execução no AWS Glue Studio. Demonstra configuração, leitura do Glue Data Catalog, controles de qualidade, consultas analíticas e geração dos dados usados nos gráficos. A ingestão parametrizada está em `../glue/processamento_consolidado.py` e os mapeamentos completos SOT/SPEC permanecem nos SQLs do repositório.

In [ ]:
# Configuração do Spark no AWS Glue
from pyspark.sql import functions as F
from pyspark.sql.window import Window

DATABASE = 'tech_challenge_db'
YEARS = [2023, 2024, 2025]
TABLES = {ano: f'{DATABASE}.spec_pesquisas_{ano}' for ano in YEARS}
print(TABLES)

## 1. Leitura da camada Gold e harmonização
A camada SPEC representa a Gold: contém nomes de negócio após a padronização realizada nas camadas anteriores. Em 2025 a chave é `token`; nos demais anos, `id`. Mudanças de formulação e roteamento entre os questionários são controladas pelo ano e pelo tamanho da amostra.

In [ ]:
def select_core(ano):
    df = spark.table(TABLES[ano])
    id_col = 'token' if ano == 2025 else 'id'
    modelo = F.col('modelo_atual_de_trabalho')
    return df.select(
        F.lit(ano).alias('ano'), F.col(id_col).cast('string').alias('id'),
        'genero', 'regiao_residencia', 'setor_atuacao', 'cargo_atual',
        'nivel_cargo_atual', 'faixa_salarial', modelo.alias('modelo_atual_de_trabalho')
    )

mercado = select_core(2023).unionByName(select_core(2024)).unionByName(select_core(2025))
mercado.cache()
mercado.groupBy('ano').count().orderBy('ano').show()

## 2. Qualidade dos dados
Os resultados devem ser registrados como evidência da execução. As contagens por tabela também ajudam a evitar conclusões baseadas em recortes muito pequenos.

In [ ]:
quality = mercado.groupBy('ano').agg(
    F.count('*').alias('linhas'),
    F.countDistinct('id').alias('ids_distintos'),
    F.sum(F.col('genero').isNull().cast('int')).alias('nulos_genero'),
    F.sum(F.col('cargo_atual').isNull().cast('int')).alias('nulos_cargo'),
    F.sum(F.col('regiao_residencia').isNull().cast('int')).alias('nulos_regiao'),
    F.sum(F.col('faixa_salarial').isNull().cast('int')).alias('nulos_salario')
).orderBy('ano')
quality.show(truncate=False)
assert quality.filter(F.col('linhas') <= 0).count() == 0, 'Há tabela Gold vazia'
assert quality.filter(F.col('ids_distintos') > F.col('linhas')).count() == 0

## 3. Indicadores de mercado
As funções abaixo calculam quantidade e percentual dentro de cada ano. O denominador é sempre explicitado para evitar comparar perguntas com populações diferentes.

In [ ]:
def distribution(df, dimension, minimum_sample=1):
    counts = (df.where(F.col(dimension).isNotNull())
              .groupBy('ano', dimension).agg(F.count('*').alias('quantidade')))
    window = Window.partitionBy('ano')
    return (counts.withColumn('base_ano', F.sum('quantidade').over(window))
            .withColumn('percentual', F.round(100 * F.col('quantidade') / F.col('base_ano'), 1))
            .where(F.col('quantidade') >= minimum_sample))

cargos = distribution(mercado, 'cargo_atual', 20)
senioridade = distribution(mercado, 'nivel_cargo_atual', 20)
setores = distribution(mercado.where(F.col('cargo_atual').isNotNull()), 'setor_atuacao', 20)
genero = distribution(mercado, 'genero', 20)
regioes = distribution(mercado, 'regiao_residencia', 20)
modelo_trabalho = distribution(mercado, 'modelo_atual_de_trabalho', 20)
cargos.orderBy('ano', F.desc('quantidade')).show(30, truncate=False)

## 4. Diversidade de gênero
O indicador usa participação feminina dentro de cada cargo ou senioridade. Recortes com menos de 20 respostas devem ser sinalizados ou removidos da apresentação.

In [ ]:
def female_share(dimension):
    return (mercado.where(F.col(dimension).isNotNull() & F.col('genero').isNotNull())
            .groupBy('ano', dimension)
            .agg(F.count('*').alias('base'),
                 F.sum(F.lower('genero').startswith('femin').cast('int')).alias('mulheres'))
            .withColumn('pct_mulheres', F.round(100 * F.col('mulheres') / F.col('base'), 1))
            .where(F.col('base') >= 20))

mulheres_cargo = female_share('cargo_atual')
mulheres_senioridade = female_share('nivel_cargo_atual')
mulheres_senioridade.orderBy('ano', 'nivel_cargo_atual').show(50, truncate=False)

## 5. Remuneração, tecnologias e IA
A remuneração usa o ponto médio da faixa como aproximação. Flags de múltipla escolha são transformadas em 0/1 e mantêm o tamanho da base no resultado. IA é comparada somente entre 2024 e 2025.

In [ ]:
faixa = F.lower(F.col('faixa_salarial'))
salario_proxy = (F.when(faixa.contains('menos de r$ 1.000'), 500)
    .when(faixa.contains('1.001') & faixa.contains('2.000'), 1500)
    .when(faixa.contains('2.001') & faixa.contains('3.000'), 2500)
    .when(faixa.contains('3.001') & faixa.contains('4.000'), 3500)
    .when(faixa.contains('4.001') & faixa.contains('6.000'), 5000)
    .when(faixa.contains('6.001') & faixa.contains('8.000'), 7000)
    .when(faixa.contains('8.001') & faixa.contains('12.000'), 10000)
    .when(faixa.contains('12.001') & faixa.contains('16.000'), 14000)
    .when(faixa.contains('16.001') & faixa.contains('20.000'), 18000)
    .when(faixa.contains('20.001') & faixa.contains('25.000'), 22500)
    .when(faixa.contains('25.001') & faixa.contains('30.000'), 27500)
    .when(faixa.contains('30.001') & faixa.contains('40.000'), 35000)
    .when(faixa.contains('acima de r$ 40.000'), 45000))
salarios = mercado.withColumn('salario_proxy', salario_proxy)
salario_senioridade = (salarios.where(F.col('salario_proxy').isNotNull())
    .groupBy('ano', 'nivel_cargo_atual').agg(F.count('*').alias('base'),
    F.round(F.avg('salario_proxy'), 0).alias('salario_medio_estimado'),
    F.expr('percentile_approx(salario_proxy, 0.5)').alias('salario_mediano_estimado'))
    .where(F.col('base') >= 20))

def flag01(name):
    value = F.lower(F.trim(F.col(name).cast('string')))
    return (F.when(value.isin('1', 'true', 'sim'), 1)
            .when(F.col(name).isNotNull(), 0))

def technology_year(ano):
    id_col = 'token' if ano == 2025 else 'id'
    return spark.table(TABLES[ano]).select(F.lit(ano).alias('ano'), F.col(id_col).alias('id'),
        flag01('flag_utiliza_sql').alias('sql'), flag01('flag_utiliza_python').alias('python'),
        flag01('flag_utiliza_aws').alias('aws'), flag01('flag_utiliza_azure').alias('azure'),
        flag01('flag_utiliza_google_cloud').alias('gcp'))

tecnologias = technology_year(2023).unionByName(technology_year(2024)).unionByName(technology_year(2025))
tecnologias_long = (tecnologias.select('ano', F.explode(F.array(
    F.struct(F.lit('SQL').alias('tecnologia'), F.col('sql').alias('adotou')),
    F.struct(F.lit('Python').alias('tecnologia'), F.col('python').alias('adotou')),
    F.struct(F.lit('AWS').alias('tecnologia'), F.col('aws').alias('adotou')),
    F.struct(F.lit('Azure').alias('tecnologia'), F.col('azure').alias('adotou')),
    F.struct(F.lit('GCP').alias('tecnologia'), F.col('gcp').alias('adotou')))).alias('item'))
    .where(F.col('item.adotou').isNotNull()))
tecnologia_adocao = (tecnologias_long.groupBy('ano', 'item.tecnologia')
    .agg(F.count('item.adotou').alias('base'), F.sum('item.adotou').alias('quantidade'))
    .withColumn('percentual', F.round(100 * F.col('quantidade') / F.col('base'), 1)))

ia_prioridade = (spark.table(TABLES[2024]).select(F.lit(2024).alias('ano'),
    F.col('ai_generativa_e_uma_prioridade_em_sua_empresa').alias('prioridade'))
    .unionByName(spark.table(TABLES[2025]).select(F.lit(2025).alias('ano'),
    F.col('ai_generativa_e_llm_e_uma_prioridade').alias('prioridade')))
    .where(F.col('prioridade').isNotNull()).groupBy('ano', 'prioridade').count())
salario_senioridade.orderBy('ano', 'salario_mediano_estimado').show(50, truncate=False)

## 6. Exportação das tabelas para os gráficos
Cada conjunto é gravado em Parquet na área analítica. Troque apenas o bucket se o laboratório utilizar outro identificador.

In [ ]:
OUTPUT = 's3://tech-challenge-018298043465/analises_executivas'
outputs = {
    'cargos': cargos, 'senioridade': senioridade, 'setores': setores,
    'genero': genero, 'regioes': regioes, 'modelo_trabalho': modelo_trabalho,
    'mulheres_cargo': mulheres_cargo, 'mulheres_senioridade': mulheres_senioridade,
    'salario_senioridade': salario_senioridade, 'tecnologia_adocao': tecnologia_adocao,
    'ia_prioridade': ia_prioridade,
    'qualidade': quality
}
for name, dataframe in outputs.items():
    dataframe.coalesce(1).write.mode('overwrite').parquet(f'{OUTPUT}/{name}/')
    print(f'Gravado: {OUTPUT}/{name}/')

## 7. Gráficos reproduzíveis
Os exemplos usam os mesmos agregados gravados no S3. Somente resultados pequenos e já agregados são convertidos para Pandas, evitando sobrecarregar o driver.

In [ ]:
import matplotlib.pyplot as plt

def comparative_bar(dataframe, category, title, value='percentual', top_n=8):
    ranked = (dataframe.withColumn('_rank', F.row_number().over(
        Window.partitionBy('ano').orderBy(F.desc(value))))
        .where(F.col('_rank') <= top_n).select('ano', category, value).toPandas())
    pivot = ranked.pivot(index=category, columns='ano', values=value).fillna(0)
    ax = pivot.plot(kind='bar', figsize=(12, 6), color=['#2563EB', '#14B8A6', '#F97316'])
    ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel('% dos respondentes')
    plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

comparative_bar(cargos, 'cargo_atual', 'Cargos mais comuns', top_n=6)
comparative_bar(senioridade, 'nivel_cargo_atual', 'Senioridade', top_n=6)
comparative_bar(regioes, 'regiao_residencia', 'Distribuição regional', top_n=8)
comparative_bar(tecnologia_adocao, 'tecnologia', 'Adoção tecnológica', top_n=8)

## Conclusão técnica
O pipeline preserva a origem, converte os dados para formato colunar, cataloga as tabelas, padroniza os campos de negócio, registra verificações de qualidade e gera conjuntos agregados reproduzíveis. As ressalvas metodológicas devem acompanhar os gráficos: redução da amostra ao longo dos anos, mudanças nas perguntas e uso do ponto médio da faixa salarial como aproximação.